# Decision Consistency Evaluation — CIFAR-10 / CIFAR-100

Measures three robustness metrics across four ImageNet-pretrained CNNs:
- **Label Stability (S):** fraction of perturbations that preserve the predicted class
- **Confidence Deviation (D):** normalised mean shift in confidence
- **Consistency Score (C):** α·S + (1−α)·(1−D)

Six non-adversarial perturbations: rotation, brightness, noise, contrast, blur, JPEG compression.

**Before running:** update the paths in the *Configuration* cell below.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Update these two paths before running.
CIFAR10_DIR  = r'E:\cifar-10-python\cifar-10-batches-py'   # required
CIFAR100_DIR = r'E:\cifar-100-python\cifar-100-python'      # optional; skipped if absent
OUTPUT_DIR   = r'E:\decision_consistency_cifar_outputs'

NUM_SAMPLES = 1000
SEED        = 42
ALPHA       = 0.5   # weight for Label Stability in Consistency Score

In [ ]:
import os, pickle, json, random, io, csv
from collections import defaultdict

import numpy as np
from scipy.stats import wilcoxon, pearsonr, spearmanr

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from torchvision import models, transforms
from PIL import Image
from tqdm import tqdm

import warnings
warnings.filterwarnings('ignore')

for sub in ['tables', 'figures/consistency', 'figures/gradcam', 'figures/perclass']:
    os.makedirs(os.path.join(OUTPUT_DIR, sub), exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
def load_cifar10_test(data_dir):
    with open(os.path.join(data_dir, 'test_batch'), 'rb') as f:
        batch = pickle.load(f, encoding='bytes')
    images = batch[b'data'].reshape(-1, 3, 32, 32)
    labels = np.array(batch[b'labels'])
    classes = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']
    return images, labels, classes


def load_cifar100_test(data_dir):
    with open(os.path.join(data_dir, 'test'), 'rb') as f:
        batch = pickle.load(f, encoding='bytes')
    images = batch[b'data'].reshape(-1, 3, 32, 32)
    labels = np.array(batch[b'fine_labels'])
    with open(os.path.join(data_dir, 'meta'), 'rb') as f:
        meta = pickle.load(f, encoding='bytes')
    classes = [n.decode() for n in meta[b'fine_label_names']]
    return images, labels, classes


X10, y10, classes10 = load_cifar10_test(CIFAR10_DIR)
print(f'CIFAR-10 test: {X10.shape}  classes: {len(classes10)}')

DATASETS = {'CIFAR-10': (X10, y10, classes10)}
if os.path.isdir(CIFAR100_DIR):
    X100, y100, classes100 = load_cifar100_test(CIFAR100_DIR)
    DATASETS['CIFAR-100'] = (X100, y100, classes100)
    print(f'CIFAR-100 test: {X100.shape}  classes: {len(classes100)}')
else:
    print('CIFAR-100 not found — running CIFAR-10 only.')

In [ ]:
def load_models(device):
    zoo = [
        ('ResNet-18',   models.resnet18),
        ('ResNet-50',   models.resnet50),
        ('VGG-16',      models.vgg16),
        ('MobileNetV2', models.mobilenet_v2),
    ]
    loaded = {}
    for name, fn in zoo:
        print(f'Loading {name}...', end=' ', flush=True)
        loaded[name] = fn(pretrained=True).eval().to(device)
        print('OK')
    return loaded

MODELS = load_models(device)
print('Models ready:', list(MODELS.keys()))

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

def prepare(img_tensor):
    pil = transforms.ToPILImage()(img_tensor.cpu().clamp(0, 1))
    return preprocess(pil).unsqueeze(0).to(device)


def apply_jpeg(img_tensor, quality=75):
    pil = transforms.ToPILImage()(img_tensor.cpu().clamp(0, 1))
    buf = io.BytesIO()
    pil.save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    return transforms.ToTensor()(Image.open(buf))


def generate_perturbations(img_tensor):
    noise = 0.01 * torch.randn_like(img_tensor)
    return {
        'rotation':   TF.rotate(img_tensor, angle=2),
        'brightness': TF.adjust_brightness(img_tensor, brightness_factor=1.1),
        'noise':      torch.clamp(img_tensor + noise, 0.0, 1.0),
        'contrast':   TF.adjust_contrast(img_tensor, contrast_factor=1.1),
        'blur':       TF.gaussian_blur(img_tensor, kernel_size=3, sigma=0.5),
        'jpeg':       apply_jpeg(img_tensor),
    }


def infer(model, img_tensor):
    with torch.no_grad():
        probs = F.softmax(model(prepare(img_tensor)), dim=1)
    return probs.argmax(dim=1).item(), probs.max().item()


def compute_metrics(orig_pred, orig_conf, pert_results, alpha=ALPHA):
    K    = len(pert_results)
    S    = sum(1 for p, _ in pert_results.values() if p == orig_pred) / K
    D    = sum(abs(c - orig_conf) / max(orig_conf, 1 - orig_conf)
               for _, c in pert_results.values()) / K
    C    = alpha * S + (1 - alpha) * (1 - D)
    return {'Label_Stability': S, 'Confidence_Deviation': D, 'Consistency_Score': C}

print('Pipeline ready.')

In [ ]:
# Main evaluation loop — expect ~10–30 min on GPU depending on hardware
results = {}

for ds_name, (X, y, class_names) in DATASETS.items():
    results[ds_name] = {}
    X_t   = torch.tensor(X, dtype=torch.float32) / 255.0
    n     = min(NUM_SAMPLES, len(y))

    for model_name, model in MODELS.items():
        per_sample = []
        for idx in tqdm(range(n), desc=f'{ds_name}/{model_name}'):
            img = X_t[idx]
            orig_pred, orig_conf = infer(model, img)
            pert = generate_perturbations(img)
            pert_results = {k: infer(model, v) for k, v in pert.items()}
            m = compute_metrics(orig_pred, orig_conf, pert_results)
            m['true_label'] = int(y[idx])
            m['class_name'] = class_names[int(y[idx])]
            m['orig_pred']  = orig_pred
            m['orig_conf']  = orig_conf
            per_sample.append(m)

        results[ds_name][model_name] = per_sample
        ls = np.mean([m['Label_Stability']      for m in per_sample])
        cd = np.mean([m['Confidence_Deviation']  for m in per_sample])
        cs = np.mean([m['Consistency_Score']     for m in per_sample])
        print(f'  [{ds_name}][{model_name}]  LS={ls:.3f}  CD={cd:.3f}  CS={cs:.3f}')

print('Evaluation complete.')

In [ ]:
# Summary table
header = ['Dataset','Model','Avg_Label_Stability','Avg_Confidence_Deviation',
          'Avg_Consistency_Score','Std_Consistency_Score']
rows = []

print(f'{"Dataset":<12} {"Model":<14} {"LS":>6} {"CD":>7} {"CS":>7} {"±CS":>7}')
print('-' * 58)

for ds_name in results:
    for model_name in results[ds_name]:
        ms  = results[ds_name][model_name]
        ls  = np.mean([m['Label_Stability']      for m in ms])
        cd  = np.mean([m['Confidence_Deviation']  for m in ms])
        cs  = np.mean([m['Consistency_Score']     for m in ms])
        std = np.std( [m['Consistency_Score']     for m in ms])
        print(f'{ds_name:<12} {model_name:<14} {ls:6.3f} {cd:7.3f} {cs:7.3f} {std:7.3f}')
        rows.append({'Dataset': ds_name, 'Model': model_name,
                     'Avg_Label_Stability':      round(ls,  4),
                     'Avg_Confidence_Deviation': round(cd,  4),
                     'Avg_Consistency_Score':    round(cs,  4),
                     'Std_Consistency_Score':    round(std, 4)})

csv_path = os.path.join(OUTPUT_DIR, 'tables', 'summary_table.csv')
with open(csv_path, 'w', newline='') as f:
    csv.DictWriter(f, fieldnames=header).writeheader()
    csv.DictWriter(f, fieldnames=header).writerows(rows)
print('Saved:', csv_path)

In [ ]:
# Statistical significance — pairwise Wilcoxon signed-rank tests
model_names = list(MODELS.keys())

for ds_name in results:
    cs_arrays = {mn: np.array([m['Consistency_Score'] for m in results[ds_name][mn]])
                 for mn in model_names if mn in results[ds_name]}
    print(f'\n=== {ds_name} ===')
    print(f'{"Pair":<30} {"W":>10} {"p":>10} {"sig":>8}')
    print('-' * 62)
    sig_rows = []
    mn_list = list(cs_arrays.keys())
    for i, mn1 in enumerate(mn_list):
        for mn2 in mn_list[i+1:]:
            w, p = wilcoxon(cs_arrays[mn1], cs_arrays[mn2])
            pair = f'{mn1} vs {mn2}'
            sig  = '***' if p < 0.05 else 'ns'
            print(f'{pair:<30} {w:>10.2f} {p:>10.4f} {sig:>8}')
            sig_rows.append({'Pair': pair, 'W': round(w,2),
                              'p_value': round(p,6), 'Significant': p < 0.05})
    path = os.path.join(OUTPUT_DIR, 'tables',
                        f'wilcoxon_{ds_name.replace("-","_")}.csv')
    with open(path, 'w', newline='') as f:
        csv.DictWriter(f, fieldnames=['Pair','W','p_value','Significant']).writeheader()
        csv.DictWriter(f, fieldnames=['Pair','W','p_value','Significant']).writerows(sig_rows)

In [ ]:
# α-ablation: verify model rankings are stable across different α values
alpha_values = [0.3, 0.4, 0.5, 0.6, 0.7]

for ds_name in results:
    fig, ax = plt.subplots(figsize=(8, 4))
    for mn, marker in zip(model_names, ['o','s','^','D']):
        if mn not in results[ds_name]:
            continue
        ms   = results[ds_name][mn]
        ls_a = np.array([m['Label_Stability']      for m in ms])
        cd_a = np.array([m['Confidence_Deviation']  for m in ms])
        vals = [float(np.mean(a*ls_a + (1-a)*(1-cd_a))) for a in alpha_values]
        ax.plot(alpha_values, vals, marker=marker, label=mn, linewidth=2, markersize=7)
    ax.set_xlabel('α', fontsize=11)
    ax.set_ylabel('Average Consistency Score', fontsize=11)
    ax.set_title(f'α-Ablation — {ds_name}\nModel rankings stable across all α', fontsize=11)
    ax.set_xticks(alpha_values)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, 'figures', 'consistency',
                        f'alpha_ablation_{ds_name.replace("-","_")}.png')
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.close()
    print('Saved:', path)

In [ ]:
# Cross-model bar chart
for ds_name in results:
    mn_list = [mn for mn in model_names if mn in results[ds_name]]
    avg_ls  = [np.mean([m['Label_Stability']      for m in results[ds_name][mn]]) for mn in mn_list]
    avg_cs  = [np.mean([m['Consistency_Score']     for m in results[ds_name][mn]]) for mn in mn_list]
    avg_cd  = [np.mean([m['Confidence_Deviation']  for m in results[ds_name][mn]]) for mn in mn_list]

    x, w = np.arange(len(mn_list)), 0.25
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(x - w, avg_ls, w, label='Label Stability',      color='steelblue')
    ax.bar(x,     avg_cs, w, label='Consistency Score',    color='darkorange')
    ax.bar(x + w, avg_cd, w, label='Confidence Deviation', color='seagreen')
    ax.set_xticks(x)
    ax.set_xticklabels(mn_list, fontsize=11)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Score')
    ax.set_title(f'Cross-Model Decision Consistency — {ds_name}', fontsize=12)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, 'figures', 'consistency',
                        f'crossmodel_{ds_name.replace("-","_")}.png')
    plt.savefig(path, dpi=300, bbox_inches='tight')
    plt.close()
    print('Saved:', path)

In [ ]:
# Per-class consistency bar charts
for ds_name in results:
    for model_name in results[ds_name]:
        ms           = results[ds_name][model_name]
        class_scores = defaultdict(list)
        for m in ms:
            class_scores[m['class_name']].append(m['Consistency_Score'])

        sorted_cls = sorted(
            {cls: np.mean(s) for cls, s in class_scores.items()}.items(),
            key=lambda x: x[1]
        )[:20]
        cls_names = [c for c, _ in sorted_cls]
        cls_vals  = [v for _, v in sorted_cls]
        colors    = ['tomato' if v < 0.5 else 'steelblue' for v in cls_vals]

        fig, ax = plt.subplots(figsize=(max(8, len(sorted_cls)*0.6), 5))
        ax.bar(cls_names, cls_vals, color=colors, edgecolor='white')
        ax.set_xticklabels(cls_names, rotation=45, ha='right', fontsize=9)
        ax.set_ylim(0, 1)
        ax.set_ylabel('Avg Consistency Score')
        ax.set_title(f'Per-Class Consistency — {ds_name} / {model_name}', fontsize=11)
        ax.axhline(0.5, color='gray', linestyle='--', lw=1)
        plt.tight_layout()
        fname = f'perclass_{ds_name}_{model_name}.png'.replace('-','_').replace(' ','_')
        path  = os.path.join(OUTPUT_DIR, 'figures', 'perclass', fname)
        plt.savefig(path, dpi=300, bbox_inches='tight')
        plt.close()
        print('Saved:', path)

In [ ]:
# Grad-CAM: qualitative visualisation for brittle samples
class GradCAM:
    def __init__(self, model, model_name):
        self.model = model
        self.gradients = self.activations = None
        target = model.layer4[-1] if 'ResNet' in model_name else model.features[-1]
        target.register_forward_hook(
            lambda m, i, o: setattr(self, 'activations', o.detach()))
        target.register_full_backward_hook(
            lambda m, gi, go: setattr(self, 'gradients', go[0].detach()))

    def generate(self, inp, class_idx):
        self.model.zero_grad()
        self.model(inp)[0, class_idx].backward()
        w   = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = F.relu((w * self.activations).sum(dim=1).squeeze()).cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return np.array(
            Image.fromarray((cam * 255).astype(np.uint8)).resize((224, 224), Image.BILINEAR)
        ) / 255.0


def denormalize(t):
    arr = t.squeeze().permute(1, 2, 0).detach().cpu().numpy()
    return np.clip(arr * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406]), 0, 1)


X_t = torch.tensor(X10, dtype=torch.float32) / 255.0

for model_name, model in MODELS.items():
    brittle_idx = next(
        (i for i, m in enumerate(results['CIFAR-10'][model_name])
         if m['Label_Stability'] < 1.0), None
    )
    if brittle_idx is None:
        print(f'{model_name}: no brittle sample found, skipping')
        continue

    img_orig = X_t[brittle_idx]
    img_pert = generate_perturbations(img_orig)['rotation']
    gcam     = GradCAM(model, model_name)
    inp_o    = prepare(img_orig).requires_grad_(True)
    inp_p    = prepare(img_pert).requires_grad_(True)
    op, _    = infer(model, img_orig)
    pp, _    = infer(model, img_pert)
    cam_o    = gcam.generate(inp_o, op)
    cam_p    = gcam.generate(inp_p, pp)

    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    for ax, inp_t, cam, title in [
        (axes[0], inp_o, cam_o, f'Original  pred:{op}'),
        (axes[1], inp_p, cam_p, f'Rotation +2°  pred:{pp}'),
    ]:
        ax.imshow(denormalize(inp_t))
        ax.imshow(cam, cmap='jet', alpha=0.45)
        ax.set_title(title, fontsize=9)
        ax.axis('off')
    fig.suptitle(f'Grad-CAM — {model_name} / CIFAR-10', fontsize=10)
    plt.tight_layout()
    fname = f'gradcam_{model_name}.png'.replace('-','_').replace(' ','_')
    plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'gradcam', fname),
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f'Saved GradCAM: {model_name}')